In [14]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import duckdb

print("🔐 Autenticando con Google...")
CONFIG_SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

# Cargar credenciales locales
credenciales = Credentials.from_service_account_file('credenciales.json', scopes=CONFIG_SCOPES)
cliente_gmail = gspread.authorize(credenciales)

# Conectar al archivo exacto
sheet_id = cliente_gmail.open('DB_Tienda_Operacional')
print("✅ Conectado exitosamente a DB_Tienda_Operacional")

🔐 Autenticando con Google...
✅ Conectado exitosamente a DB_Tienda_Operacional


In [15]:
print("📥 1/2: Extrayendo tablas desde Google Sheets...")
df_ventas = pd.DataFrame(sheet_id.worksheet('Ventas').get_all_records())
df_detalle = pd.DataFrame(sheet_id.worksheet('Detalle_Ventas').get_all_records())
df_productos = pd.DataFrame(sheet_id.worksheet('Productos').get_all_records())
df_gastos = pd.DataFrame(sheet_id.worksheet('Gastos').get_all_records())

print("🧹 2/2: Curando los datos (forzando formato numérico)...")
df_detalle['cantidad'] = pd.to_numeric(df_detalle['cantidad'], errors='coerce').fillna(0)
df_detalle['precio_compra_aplicado'] = pd.to_numeric(df_detalle['precio_compra_aplicado'], errors='coerce').fillna(0)
df_detalle['precio_venta_aplicado'] = pd.to_numeric(df_detalle['precio_venta_aplicado'], errors='coerce').fillna(0)

print("✅ Datos extraídos y limpios en memoria.")

📥 1/2: Extrayendo tablas desde Google Sheets...
🧹 2/2: Curando los datos (forzando formato numérico)...
✅ Datos extraídos y limpios en memoria.


In [16]:
print("⚡ Cruzando datos con motor SQL en memoria (DuckDB)...")

query_ventas_completas = """
    SELECT 
        v.fecha_hora,
        v.metodo_pago,
        d.id_producto,
        p.nombre,
        d.cantidad,
        d.precio_compra_aplicado AS precio_compra,
        d.precio_venta_aplicado AS precio_venta,
        (d.cantidad * d.precio_venta_aplicado) AS subtotal,
        (d.precio_venta_aplicado - d.precio_compra_aplicado) * d.cantidad AS ganancia_bruta
    FROM df_ventas AS v
    JOIN df_detalle AS d ON v.id_venta = d.id_venta
    JOIN df_productos AS p ON d.id_producto = p.id_producto
"""

# Ejecutar y convertir a DataFrame de Pandas
df_modelo_ventas = duckdb.query(query_ventas_completas).to_df()

# Limpiar posibles nulos post-JOIN para que Sheets no se queje
df_modelo_ventas = df_modelo_ventas.fillna("")

print("✅ Transformación exitosa. Muestra de los datos:")
display(df_modelo_ventas.head())

⚡ Cruzando datos con motor SQL en memoria (DuckDB)...
✅ Transformación exitosa. Muestra de los datos:


,fecha_hora,metodo_pago,id_producto,nombre,cantidad,precio_compra,precio_venta,subtotal,ganancia_bruta
0,01/01/2009 7:07:07,QR,9902f327,Coca Cola 600ml,2,5,8,16,6
1,23/03/2020 7:08:15,Efectivo,9902f327,Coca Cola 600ml,3,5,8,24,9


In [17]:
print("📤 Preparando carga a la capa analítica...")

# 1. Gestionar la pestaña destino
try:
    hoja_destino = sheet_id.add_worksheet(title="BI_Ventas_Modeladas", rows="1000", cols="20")
    print("✨ Pestaña 'BI_Ventas_Modeladas' creada desde cero.")
except Exception:
    hoja_destino = sheet_id.worksheet("BI_Ventas_Modeladas")
    hoja_destino.clear()
    print("♻️ Pestaña existente limpiada con éxito.")

# 2. Formatear para Google Sheets (Lista de listas)
columnas = df_modelo_ventas.columns.values.tolist()
filas = df_modelo_ventas.values.tolist()
datos_a_subir = [columnas] + filas

# 3. Subir datos
hoja_destino.update(range_name='A1', values=datos_a_subir)
print("🚀 ¡ETL COMPLETADO! Dashboard listo para consumir los nuevos datos.")

📤 Preparando carga a la capa analítica...
♻️ Pestaña existente limpiada con éxito.
🚀 ¡ETL COMPLETADO! Dashboard listo para consumir los nuevos datos.
